# RAG 데이터 확인용 노트북

벡터스토어에 저장된 청크, 하이브리드 검색 결과, DART XML 파싱 결과를 셀 단위로 확인하기 위한 노트북입니다.
커널은 `RAG_project (.venv)`를 선택하세요.

In [ ]:
import pandas as pd
from rag_core import vectorstore, embeddings, llm
from dart_parser import parse_dart_xml
from hybrid_search import hybrid_search, _detect_company_filter

pd.set_option("display.max_colwidth", 120)

## 1. 저장된 청크 전체 개요
문서(source)별로 몇 개의 청크가 들어있는지 확인합니다.

In [ ]:
raw = vectorstore.get(include=["documents", "metadatas"])
df = pd.DataFrame({
    "id": raw["ids"],
    "text": raw["documents"],
    **{k: [m.get(k) for m in raw["metadatas"]] for k in ["source", "chunk_index", "section", "company", "doc_type", "fiscal_period", "ceo_name"]},
})
print("총 청크 수:", len(df))
df.groupby("source").size().sort_values(ascending=False)

## 2. 특정 문서의 청크 살펴보기
`source_filter`를 원하는 파일명으로 바꿔서 실행하세요. `text`에 회사명/대표이사/사업연도/섹션이 담긴 contextual header가 맨 앞줄에 붙어있는 걸 확인할 수 있습니다.

In [ ]:
source_filter = df["source"].iloc[0] if len(df) else None
df[df["source"] == source_filter][["chunk_index", "section", "ceo_name", "text"]]

## 3. 하이브리드 검색 테스트
질문을 바꿔가며 어떤 청크가 검색되는지 확인합니다. **꼭 `hybrid_search()`를 써야 회사명 자동 필터링 + BM25 키워드 검색이 적용됩니다.** (`vectorstore.similarity_search`를 직접 쓰면 의미 검색만 되고 이 프로젝트에서 겪었던 문제들이 그대로 재현됩니다.) 질문에 저장된 회사명이 들어있으면 자동으로 그 회사로 범위를 좁혀서 검색하고, `감지된 회사 필터`로 확인할 수 있습니다.

In [ ]:
question = "삼양애니팜 대표이사가 누구야?"
print("감지된 회사 필터:", _detect_company_filter(question))

docs = hybrid_search(question, k=5)
pd.DataFrame([
    {
        "source": doc.metadata.get("source"),
        "company": doc.metadata.get("company"),
        "section": doc.metadata.get("section"),
        "text": doc.page_content[:200],
    }
    for doc in docs
])

In [ ]:
# 참고용: 벡터(의미) 검색만 썼을 때는 어떻게 다른지 비교 (하이브리드 검색과 결과가 다를 수 있음)
results = vectorstore.similarity_search_with_score(question, k=5)

pd.DataFrame([
    {
        "score": score,
        "source": doc.metadata.get("source"),
        "section": doc.metadata.get("section"),
        "text": doc.page_content[:200],
    }
    for doc, score in results
])

## 4. 전체 RAG 파이프라인(query.py의 ask) 실행
검색 + 프롬프트 + GPT 답변까지 한 번에 확인합니다. `query.py`의 `ask()`는 내부적으로 `hybrid_search()`를 사용합니다.

In [ ]:
from query import ask

print(ask(question))

## 5. (선택) DART XML 파서 결과만 따로 확인
`ingest.py`를 거치기 전, `dart_parser`가 특정 XML 파일을 어떤 섹션들로 쪼개는지, `ceo_name`/`fiscal_period`를 정확히 뽑는지 미리 확인할 때 사용합니다. 여기서 보이는 `preview`에는 아직 contextual header가 안 붙어있습니다 (그건 `ingest.py`가 청크 분할 이후에 붙이는 것이라서).

In [ ]:
from pathlib import Path

xml_path = Path("data/dart_xml/20200814000905.xml")  # 확인하고 싶은 파일로 변경
sections = parse_dart_xml(xml_path)

print("company:", sections[0].metadata["company"], "| ceo_name:", sections[0].metadata["ceo_name"], "| fiscal_period:", sections[0].metadata["fiscal_period"])
pd.DataFrame([
    {"section": d.metadata["section"], "length": len(d.page_content), "preview": d.page_content[:150]}
    for d in sections
])